
# การลดขนาดแบบกำเนิด

การลดขนาดแบบสร้างทั่วไต้หวันโดยใช้โมเดลการแพร่กระจาย CorrDiff

ตัวอย่างนี้จะสาธิตวิธีการใช้โมเดล CorrDiff ของ Nvidia ที่ได้รับการฝึกอบรมมา
ทำนายสภาพอากาศทั่วไต้หวัน เพื่อลดขนาดลงจากระดับไตรมาส
ข้อมูล forecast ทั่วโลกถึง ~ 3 กม.

checkpoint นี้ได้รับการฝึกอบรมเกี่ยวกับข้อมูล ERA5 และข้อมูล WRF ที่ครอบคลุมช่วงปี 2018-2021 ในหนึ่งชั่วโมง
ความละเอียดของเวลา ในตัวอย่างนี้ เราสาธิตการประยุกต์ใช้กับข้อมูล GFS สำหรับพายุไต้ฝุ่น
ความละเอียดสุดยอดตั้งแต่ปี 2023 ประสิทธิภาพของโมเดลกับข้อมูล GFS และข้อมูลจากปีนี้
ยังไม่ได้รับการประเมิน

ในตัวอย่างนี้คุณจะได้เรียนรู้:

- การสร้าง workflow แบบกำหนดเองสำหรับการรัน CorrDiff inference
- การสร้างแหล่งข้อมูลสำหรับอินพุตของ CorrDiff
- การเริ่มต้นและการรัน CorrDiff Diagnostic Model
- ผลลัพธ์หลังการประมวลผล


In [ ]:
# /// script
# dependencies = [
#   "earth2studio[corrdiff] @ git+https://github.com/NVIDIA/earth2studio.git",
#   "cartopy",
# ]
# ///

## การสร้างเวิร์กโฟลว์ CorrDiff อย่างง่าย

ตามปกติ เราเริ่มต้นด้วยการสร้าง workflow ง่ายๆ เพื่อรัน CorrDiff เพื่อเพิ่ม
ลักษณะทั่วไปของ workflow นี้ เราใช้การฉีดพึ่งพาตามรูปแบบ
ให้ไว้ภายใน :py:obj:`earth2studio.run` เนื่องจาก CorrDiff เป็น Diagnostic Model สิ่งนี้
workflow จะไม่ทำนายอนุกรมเวลา แต่จะเป็นการทำนายแบบทันทีทันใด


สำหรับ workflow นี้ เราระบุ

- time: รายการ datetime / string ที่ต้องการใช้รัน inference
- Corrdiff: โมเดล CorrDiffTaiwan เริ่มต้น
- data: data source ที่เตรียมไว้แล้วสำหรับดึง initial conditions
- io: IOBackend
- number_of_samples: จำนวนตัวอย่างที่จะสร้างจากโมเดล



In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()  # สิ่งที่ต้องทำ: สร้างฟังก์ชันการเตรียมตัวอย่างทั่วไป

from collections import OrderedDict
from datetime import datetime

import numpy as np
import torch
from loguru import logger

from earth2studio.data import DataSource, prep_data_array
from earth2studio.io import IOBackend
from earth2studio.models.dx import CorrDiffTaiwan
from earth2studio.utils.coords import map_coords, split_coords
from earth2studio.utils.time import to_time_array


def run(
    time: list[str] | list[datetime] | list[np.datetime64],
    corrdiff: CorrDiffTaiwan,
    data: DataSource,
    io: IOBackend,
    number_of_samples: int = 1,
) -> IOBackend:
    """CorrDiff infernce workflow

    Parameters
    ----------
    time : list[str] | list[datetime] | list[np.datetime64]
        List of string, datetimes or np.datetime64
    corrdiff : CorrDiffTaiwan
        CorrDiff mode
    data : DataSource
        Data source
    io : IOBackend
        IO object
    number_of_samples : int, optional
        Number of samples to generate, by default 1

    Returns
    -------
    IOBackend
        Output IO object
    """
    logger.info("Running corrdiff inference!")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Inference device: {device}")

    corrdiff = corrdiff.to(device)
    # อัปเดตจำนวนตัวอย่างสำหรับ Corrdiff ที่จะสร้าง
    corrdiff.number_of_samples = number_of_samples

    # ดึงข้อมูลจากแหล่งข้อมูลและโหลดลงในอุปกรณ์
    time = to_time_array(time)
    x, coords = prep_data_array(
        data(time, corrdiff.input_coords()["variable"]), device=device
    )
    x, coords = map_coords(x, coords, corrdiff.input_coords())

    logger.success(f"Fetched data from {data.__class__.__name__}")

    # ตั้งค่า IO Backend
    output_coords = corrdiff.output_coords(corrdiff.input_coords())
    total_coords = OrderedDict(
        {
            "time": coords["time"],
            "sample": output_coords["sample"],
            "lat": output_coords["lat"],
            "lon": output_coords["lon"],
        }
    )
    io.add_array(total_coords, output_coords["variable"])

    logger.info("Inference starting!")
    x, coords = corrdiff(x, coords)
    io.write(*split_coords(x, coords))

    logger.success("Inference complete")
    return io

## การเตรียมองค์ประกอบ
ด้วยการกำหนด workflow ขั้นตอนต่อไปคือการเริ่มต้นส่วนประกอบที่จำเป็น
สตูดิโอเอิร์ธ-2

ชัดเจนว่าเราต้องการสิ่งต่อไปนี้:

- Diagnostic Model: รุ่น CorrDiff สำหรับไต้หวัน :py:class:`earth2studio.models.dx.CorrDiffTaiwan`
- Datasource: ดึงข้อมูลจาก GFS data API ผ่าน :py:class:`earth2studio.data.GFS`.
- IO Backend: บันทึกเอาต์พุตลงในร้านค้า Zarr :py:class:`earth2studio.io.ZarrBackend`




In [ ]:
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend

# สร้างโมเดล CorrDiff
package = CorrDiffTaiwan.load_default_package()
corrdiff = CorrDiffTaiwan.load_model(package)

# สร้างแหล่งข้อมูล
data = GFS()

# สร้างตัวจัดการ IO เก็บไว้ในหน่วยความจำ
io = ZarrBackend()

## การรัน Workflow
เมื่อคอมโพเนนต์ทั้งหมดเริ่มต้นแล้ว การรัน workflow จะเป็นโค้ด Python บรรทัดเดียว
เวิร์กโฟลว์จะส่งคืนอ็อบเจ็กต์ IO ที่ระบุกลับไปยังผู้ใช้ ซึ่งสามารถนำมาใช้ได้
จากนั้นจึงโพสต์กระบวนการ บางตัวมี API เพิ่มเติมซึ่งมีประโยชน์สำหรับ post-processing หรือ
บันทึกเป็นไฟล์ ตรวจสอบเอกสาร API สำหรับข้อมูลเพิ่มเติม

สำหรับ inference เราจะคาดการณ์ 1 ตัวอย่างสำหรับการประทับเวลาเฉพาะที่เป็นตัวแทน
พายุไต้ฝุ่นโคอินุ



In [ ]:
io = run(["2023-10-04T18:00:00"], corrdiff, data, io, number_of_samples=1)

## การทำ Post-Processing
ขั้นตอนสุดท้ายคือการนำผลลัพธ์มาทำ post-process ต่อ Cartopy เป็นไลบรารีที่เหมาะมากสำหรับการพล็อตฟิลด์ข้อมูลบน projection ของทรงกลม

สังเกตว่า Zarr IO function มี API เพิ่มเติมสำหรับใช้เข้าถึงและจัดการข้อมูลที่จัดเก็บไว้



In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

projection = ccrs.LambertConformal(
    central_longitude=io["lon"][:].mean(),
)

fig = plt.figure(figsize=(4 * 8, 8))

ax0 = fig.add_subplot(1, 3, 1, projection=projection)
c = ax0.pcolormesh(
    io["lon"],
    io["lat"],
    io["mrr"][0, 0],
    transform=ccrs.PlateCarree(),
    cmap="inferno",
)
plt.colorbar(c, ax=ax0, shrink=0.6, label="mrr dBz")
ax0.coastlines()
ax0.gridlines()
ax0.set_title("Radar Reflectivity")

ax1 = fig.add_subplot(1, 3, 2, projection=projection)
c = ax1.pcolormesh(
    io["lon"],
    io["lat"],
    io["t2m"][0, 0],
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
)
plt.colorbar(c, ax=ax1, shrink=0.6, label="K")
ax1.coastlines()
ax1.gridlines()
ax1.set_title("2-meter Temperature")

ax2 = fig.add_subplot(1, 3, 3, projection=projection)
c = ax2.pcolormesh(
    io["lon"],
    io["lat"],
    np.sqrt(io["u10m"][0, 0] ** 2 + io["v10m"][0, 0] ** 2),
    transform=ccrs.PlateCarree(),
    cmap="Greens",
)
plt.colorbar(c, ax=ax2, shrink=0.6, label="w10m m s^-1")
ax2.coastlines()
ax2.gridlines()
ax2.set_title("10-meter Wind Speed")

plt.savefig("outputs/04_corr_diff_prediction.jpg")